In [31]:
import torch

In [42]:
class NeuralNet():
    def __init__(self, input_features, hidden_layer1, hidden_layer2, bias, learning_rate):
        self.learning_rate = learning_rate

        self.W1 = torch.randn((input_features, hidden_layer1), dtype=torch.float32) * (2 / input_features) ** 0.5
        self.W2 = torch.randn((hidden_layer1, hidden_layer2), dtype=torch.float32) * (2 / hidden_layer1) ** 0.5
        self.W3 = torch.randn((hidden_layer2, 1), dtype=torch.float32) * (2 / hidden_layer2) ** 0.5


        self.b1 = torch.full((hidden_layer1, ), bias, dtype=torch.float32)
        self.b2 = torch.full((hidden_layer2, ), bias, dtype=torch.float32)
        self.b3 = torch.full((1, ), bias, dtype=torch.float32)
    
    def train(self, X_train, y_train, X_val, y_val, iterations = 100, show_every = 100):

        for iter in range(iterations):  
            train_predictions, train_cache = self.forward(X_train)
            train_loss = self.loss(train_predictions, y_train)

            val_predictions, _ = self.forward(X_val)
            val_loss = self.loss(val_predictions, y_val)
            
            if iter % show_every == 0:
                print(f"iteration: {iter}: train loss: {train_loss}, val loss: {val_loss}")
            
            self.backward(X_train, y_train, train_predictions, train_cache)


    def forward(self, X_train):
        H1 = X_train @ self.W1 + self.b1
        A1 = self.relu(H1)

        H2 = A1 @ self.W2 + self.b2
        A2 = self.relu(H2)

        H3 = A2 @ self.W3 + self.b3

        cache = (H1, A1, H2, A2, H3)

        # print(f"H3 shape: {H3.shape}")
        
        return H3, cache

    def backward(self, X, y, pred, cache):
        H1, A1, H2, A2, H3 = cache
        # print(f"y shape: {y.shape}")
        dH3 = self.loss_grad(H3, y) # shape: (N, 1)
        # print(f"dh3 shape: {dH3.shape}")

        dW3 = A2.T @ dH3  # (hidden_layer2, N) @ (N, 1) = (hidden_layer2, 1)
        db3 = torch.sum(dH3, dim = 0) # shape: (1, )
        # print(dH3.shape)
        # print(self.W3.shape)
        dA2 = dH3 @ self.W3.T # (N, 1) @ (1, hidden_layer2) = (N, hidden_layer2)

        dH2 = self.relu_grad(H2) * dA2 # (N, hidden_layer2)

        dW2 = A1.T @ dH2 # (hidden_layer1, N) @ (N, hidden_layer2) = (hidden_layer1, hidden_layer2)
        db2 = torch.sum(dH2, dim = 0) # (hidden_layer2)
        dA1 = dH2 @ self.W2.T # (N, hidden_layer2) @ (hidden_layer2, hidden_layer1) = (N, hidden_layer1)

        dH1 = self.relu_grad(H1) * dA1 # (N, hidden_layer1)

        dW1 = X.T @ dH1 # (input_featues, N) @ (N, hidden_layer1) = (input_features, hidden_layer1)
        db1 = torch.sum(dH1, dim = 0) # (hidden_layer, 1)

        self.W1 = self.update_parameters(self.W1, dW1, self.learning_rate)
        self.W2 = self.update_parameters(self.W2, dW2, self.learning_rate)
        self.W3 = self.update_parameters(self.W3, dW3, self.learning_rate)

        self.b1 = self.update_parameters(self.b1, db1, self.learning_rate)
        self.b2 = self.update_parameters(self.b2, db2, self.learning_rate)
        self.b3 = self.update_parameters(self.b3, db3, self.learning_rate)

    def loss(self, pred, y):
        N = y.shape[0]
        return torch.sum((pred - y) ** 2) / N
    
    def loss_grad(self, pred, y):
        N = y.shape[0]

        return (2 / N) * (pred - y)

    def relu(self, r):
        return torch.relu(r).float()
    
    def relu_grad(self, r):
        return torch.where(r > 0, 1, 0).float()
    
    def update_parameters(self, param, grad, learning_rate, strategy = 'simple'):
        if strategy == "simple":
            return self.simple_update(param, grad, learning_rate)
    
    def simple_update(self, param, grad, learning_rate):
        new_param = param - learning_rate * grad
        return new_param

In [12]:
# (10,000, 100) @ (100, 256) = (10,000, 256) @ (256, 512) = (10,000, 512) @ (512, 1) = (10,000, 1)

In [ ]:
# from sklearn.datasets import make_regression
# import torch

# X, y = make_regression(n_samples=500, n_features=8, noise=15.0, random_state=42)
# X = torch.tensor(X, dtype=torch.float32)
# y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
# # optional: train/val split
# n = len(X)
# split = int(0.8 * n)
# X_train, X_val = X[:split], X[split:]
# y_train, y_val = y[:split], y[split:]

In [47]:
from sklearn.datasets import load_diabetes
import torch
from sklearn.model_selection import train_test_split

data = load_diabetes()
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size = 0.2)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size = 0.2)

In [ ]:
NN = NeuralNet(X_train.shape[1], 256, 256, 0.1, 2e-5)
NN.train(X_train, y_train, X_val, y_val, iterations = 20000, show_every = 100)

iteration: 0: train loss: 28853.189453125, val loss: 27566.509765625
iteration: 100: train loss: 25014.794921875, val loss: 23844.306640625
iteration: 200: train loss: 5727.2099609375, val loss: 5778.359375
iteration: 300: train loss: 5568.05908203125, val loss: 5620.046875
iteration: 400: train loss: 5411.73046875, val loss: 5456.7060546875
iteration: 500: train loss: 5251.28955078125, val loss: 5290.234375
iteration: 600: train loss: 5083.6552734375, val loss: 5118.10986328125
iteration: 700: train loss: 4906.37255859375, val loss: 4938.07958984375
iteration: 800: train loss: 4717.478515625, val loss: 4748.029296875
iteration: 900: train loss: 4516.11767578125, val loss: 4548.54052734375
iteration: 1000: train loss: 4303.3671875, val loss: 4341.8154296875
iteration: 1100: train loss: 4084.572265625, val loss: 4134.57666015625
iteration: 1200: train loss: 3869.75, val loss: 3938.029052734375
iteration: 1300: train loss: 3671.35302734375, val loss: 3764.483154296875
iteration: 1400: tr